In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
quick_test=False
NSIM = 10000

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Circle
from scipy.interpolate import interp1d
import pandas as pd
import sncosmo
from scipy import stats
from scipy.special import expit
from nested_pandas import read_parquet
from joblib import Parallel, delayed
import cloudpickle as pickle
from regions import RectangleSkyRegion
from astropy.coordinates import SkyCoord
import astropy.units as u
from scipy.optimize import curve_fit

from lightcurvelynx.obstable.ztf_obstable import ZTFObsTable
from lightcurvelynx.astro_utils.passbands import PassbandGroup
from lightcurvelynx.astro_utils.pzflow_node import PZFlowNode
from lightcurvelynx.astro_utils.snia_utils import (
    DistModFromRedshift,
    HostmassX1Func,
    X0FromDistMod,
    num_snia_per_redshift_bin,
    SNCoordGivenPhysicalSep,
)
from lightcurvelynx.math_nodes.scipy_random import SamplePDF
from lightcurvelynx.math_nodes.np_random import NumpyRandomFunc
from lightcurvelynx.simulate import simulate_lightcurves
from lightcurvelynx.models.sncomso_models import SncosmoWrapperModel
from lightcurvelynx.models.snia_host import SNIaHost
from lightcurvelynx.utils.plotting import plot_lightcurves
from lightcurvelynx.math_nodes.ra_dec_sampler import ObsTableUniformRADECSampler
from lightcurvelynx.astro_utils.dustmap import DustmapWrapper,SFDMap
from lightcurvelynx.effects.extinction import ExtinctionEffect
from lightcurvelynx.astro_utils.mag_flux import mag2flux,flux2mag
from lightcurvelynx.astro_utils.detector_footprint import DetectorFootprint

from lightcurvelynx import _LIGHTCURVELYNX_BASE_DATA_DIR

from lightcurvelynx.validation.lcfit import fit_single_lc

In [ ]:
globalhostdata = pd.read_csv('ztfsniadr2/tables/globalhost_data.csv')
localhostdata = pd.read_csv('ztfsniadr2/tables/localhost_data.csv')
sndata = pd.read_csv('ztfsniadr2/tables/snia_data.csv')
data = pd.merge(sndata,globalhostdata,on='ztfname')

In [ ]:
lcdata = read_parquet('data/ztfsniadr2.parquet')

In [ ]:
%%time

obs_log = pd.read_parquet('data/ztf_observing_log_combined_w_metadata.parquet')
colmap = {"ra":"ra",
          "dec":"dec",
          "time":"mjd",
          "zp":"zp_nJy",
          "filter":"filter",
          "sky":"scibckgnd",
         }

#ztf ccd size 6144 × 6160 pixel * 16
pixel_scale = 1.01 #arcsec/pixel
center = SkyCoord(ra=0.0, dec=0.0, unit="deg", frame="icrs")
rect_region = RectangleSkyRegion(center=center, width=4*6144.* pixel_scale * u.arcsec, 
                                 height=4*6160.* pixel_scale * u.arcsec, angle=0.0 * u.deg)
ztf_fp = DetectorFootprint(rect_region, pixel_scale=pixel_scale)

ztf_obstable = ZTFObsTable(obs_log,colmap=colmap,detector_footprint=ztf_fp)
ztf_obstable.survey_values["zp_err_mag"] = 0.001

t_min, t_max = ztf_obstable.time_bounds()
print(f"Loaded OpSim with {len(ztf_obstable)} rows and times [{t_min}, {t_max}]")

passband_group = PassbandGroup.from_preset(preset="ZTF", filters=["g", "r", "i"])
print(f"Loaded Passbands: {passband_group}")

In [ ]:
radec_node = ObsTableUniformRADECSampler(ztf_obstable, node_label="radec")

zmin = 0.001
zmax = 0.2
H0 = 70.0
Omega_m = 0.3
nsn, z = num_snia_per_redshift_bin(zmin, zmax, 100, H0=H0, Omega_m=Omega_m)
zpdf = interp1d(z, nsn, bounds_error=False, fill_value=0)

host = SNIaHost(
    ra = radec_node.ra,
    dec = radec_node.dec,
    hostmass=10.,
    redshift=SamplePDF(zpdf),
    node_label="host",
)

In [ ]:
distmod_func = DistModFromRedshift(host.redshift, H0=H0, Omega_m=Omega_m)
x1_func = NumpyRandomFunc("uniform",low=-5.,high=5.)
c_func = NumpyRandomFunc("uniform",low=-0.5,high=1.)
m_abs_func = NumpyRandomFunc("normal", loc=-19.05, scale=0.1)

# we model host-sn separation as an exponential distribution based on Fig 3 of Gupta et al 2016, mean separation = 5kpc
physical_host_sn_sep = NumpyRandomFunc("exponential", scale = 5.)
sncoor_node = SNCoordGivenPhysicalSep(host.ra, host.dec, physical_host_sn_sep, host.redshift, H0=H0, Omega_m=Omega_m,node_label='sncoor_node')

x0_func = X0FromDistMod(
    distmod=distmod_func,
    x1=x1_func,
    c=c_func,
    alpha=0.14,
    beta=3.1,
    m_abs=m_abs_func,
    node_label="x0_func",
)

sncosmo_modelname = "salt3"
source = SncosmoWrapperModel(
    sncosmo_modelname,
    t0=NumpyRandomFunc("uniform", low=t_min, high=t_max),
    x0=x0_func,
    x1=x1_func,
    c=c_func,
    ra=sncoor_node.ra,
    dec=sncoor_node.dec,
    redshift=host.redshift,
    node_label="source",
)
    
mwextinction = SFDMap(
    ra=source.ra,
    dec=source.dec,
    node_label="mwext",
)

# Create an extinction effect using the EBVs from that dust map.
ext_effect = ExtinctionEffect(extinction_model="F99", frame='observer', ebv=mwextinction, Rv=3.1)
source.add_effect(ext_effect)


In [ ]:
%%time
if quick_test:
    nsntotal = NSIM
else:
    survey_length = (t_max - t_min)/365.
    nsntotal, _ = num_snia_per_redshift_bin(zmin, zmax, 1, H0=H0, Omega_m=Omega_m, solid_angle=9.136*survey_length*2)
    print(f"Survey length = {survey_length} years")
print(f"Simulating {int(nsntotal)} SN ...")
lightcurves = simulate_lightcurves(source, int(nsntotal), ztf_obstable, passband_group,
                                   param_cols = ['source.x0','source.x1','source.c','host.hostmass'])
lightcurves

In [ ]:
lightcurves['params'][0].keys()

In [ ]:
lightcurves.lightcurve.isna().sum()

In [ ]:
# calculate detection flag
lightcurves = lightcurves.dropna(subset=['lightcurve'])
print("Before applying detection: nsn=", len(lightcurves))
lightcurves['lightcurve.snr'] = lightcurves['lightcurve.flux']/lightcurves['lightcurve.fluxerr']
detection_snr_thres = 5.
lightcurves['lightcurve.detection_flag'] = lightcurves['lightcurve.snr'] > detection_snr_thres

lightcurves_after_detection = lightcurves.query("lightcurve.detection_flag == True").dropna()
print("After applying detection: nsn=", len(lightcurves_after_detection))

In [ ]:
# apply spec selection function

def spec_selection_func(flux,m0=18.8,s=4.5):
    m = flux2mag(np.max(flux))
    p = np.power(1. + np.exp((m - m0)*s), -1)
    p0 = np.random.uniform(0,1)
    if p0 < p:
        return {"pass_spec_selection":True}
    else:
        return {"pass_spec_selection":False}

pass_selection = lightcurves_after_detection.reduce(spec_selection_func,"lightcurve.flux")
idx = pass_selection.query("pass_spec_selection == True").index
lightcurves_after_spec_selection = lightcurves_after_detection.loc[idx]
print("After spectroscopic selection: nsn=", len(lightcurves_after_spec_selection))

lightcurves["pass_spec_selection"] = False
lightcurves.loc[idx,"pass_spec_selection"] = True

In [ ]:
def quality_cuts(flux,mjd,filter,n_phases=7, n_before_peak=2, n_after_peak=2, n_bands=2):
    peak_idx = np.argmax(flux)
    pass_cut = len(np.unique(np.floor(mjd))) >= n_phases
    pass_cut &= (peak_idx >= n_before_peak - 1) & (len(flux) - peak_idx >= n_after_peak - 1)
    pass_cut &= len(np.unique(filter)) >= n_bands
    return {"pass_quality_cuts": pass_cut}

In [ ]:
pass_quality_cut = lightcurves_after_spec_selection.reduce(quality_cuts,"lightcurve.flux",
                                                           "lightcurve.mjd","lightcurve.filter")
idx = pass_quality_cut.query("pass_quality_cuts == True").index
lightcurves_after_quality_cut = lightcurves_after_spec_selection.loc[idx]
print("After quality cuts: nsn=", len(lightcurves_after_quality_cut))

lightcurves["pass_quality_cuts"] = False
lightcurves.loc[idx,"pass_quality_cuts"] = True

In [ ]:
sim_all_x1 = lightcurves['source_x1']
sim_all_c = lightcurves['source_c']

In [ ]:
sim_x1 = lightcurves_after_quality_cut['source_x1']
sim_c = lightcurves_after_quality_cut['source_c']

In [ ]:
bins=np.linspace(-5,5,30)
plt.hist(sim_x1,bins=bins,alpha=0.3,density=True,label='sim_after_selection')
plt.hist(sim_all_x1,bins=bins,alpha=0.5,density=True,label='sim_all',histtype='step')
plt.legend()
plt.title('x1')

In [ ]:
bins=np.linspace(-0.5,1,30)
plt.hist(sim_c,bins=bins,alpha=0.3,density=True,label='sim')
plt.hist(sim_all_c,bins=bins,alpha=0.5,density=True,label='sim_all',histtype='step')
plt.legend()
plt.title('c')

In [ ]:
def expo(x, a, b, c):
    x = np.asarray(x)
    y = a * np.exp(b * x) + c
    return y

In [ ]:
bins = np.linspace(-5,5,30)
x1before,bin_edges,_ = stats.binned_statistic(sim_all_x1,np.ones(len(lightcurves)), statistic='sum', bins=bins)
x1after,bin_edges,_ =  stats.binned_statistic(sim_x1, np.ones(len(lightcurves_after_quality_cut)), statistic='sum', bins=bins)
x_x1 = (bin_edges[:-1] + bin_edges[1:])/2.
f_x1 = x1after/x1before
plt.plot(x_x1,f_x1)
outarr = np.array([x_x1,f_x1])

#fit for an exponential curve using curve_fit
popt, pcov = curve_fit(expo, x_x1, f_x1, p0=[1., 1. ,0.], bounds = ([0.0, 0.0, -np.inf], [np.inf, np.inf, np.inf]))
print(popt)
x = np.linspace(x_x1.min(),x_x1.max(),50)
plt.plot(x, expo(x, *popt))
np.savetxt('data/ztf_selection_func_x1.txt',popt)

In [ ]:
bins = np.linspace(-0.5,1,30)
cbefore,bin_edges,_ = stats.binned_statistic(sim_all_c,np.ones(len(lightcurves)), statistic='sum', bins=bins)
cafter,bin_edges,_ =  stats.binned_statistic(sim_c, np.ones(len(lightcurves_after_quality_cut)), statistic='sum', bins=bins)
x_c = (bin_edges[:-1] + bin_edges[1:])/2.
f_c = cafter/cbefore
plt.plot(x_c,f_c)
outarr = np.array([x_c,f_c])

#fit for an exponential curve using curve_fit
popt, pcov = curve_fit(expo, x_c, f_c, p0=[1., -1. ,0.], bounds = ([0.0, -np.inf, -np.inf], [np.inf, 0.0, np.inf]))
print(popt)
x = np.linspace(x_c.min(),x_c.max(),50)
plt.plot(x, expo(x, *popt))
np.savetxt('data/ztf_selection_func_c.txt',popt)